# Hello World - Jupyter Notebook Demo

Simple chat completion using LlamaStack from a Jupyter notebook.

In [ ]:
import os
import requests
from pathlib import Path
from openai import OpenAI

In [ ]:
# Load configuration from file
import yaml

config_file = Path.home() / '.lls_showroom_generated'
config = {}

if config_file.exists():
    with open(config_file, 'r') as f:
        data = yaml.safe_load(f)
        config = data.get('secrets', {})

llamastack_url = config.get('LLAMASTACK_URL', os.environ.get('LLAMASTACK_URL'))
keycloak_url = config.get('KEYCLOAK_URL', os.environ.get('KEYCLOAK_URL'))
username = config.get('KEYCLOAK_USERNAME', os.environ.get('KEYCLOAK_USERNAME'))
password = config.get('KEYCLOAK_PASSWORD', os.environ.get('KEYCLOAK_PASSWORD'))
client_secret = config.get('KEYCLOAK_CLIENT_SECRET', os.environ.get('KEYCLOAK_CLIENT_SECRET'))

print(f"Connecting to: {llamastack_url}")

In [ ]:
# Get authentication token if Keycloak is configured
api_key = "not-needed"

if keycloak_url and username and password and client_secret:
    print(f"\n🔐 Authenticating with Keycloak as '{username}'...")
    
    token_url = f"{keycloak_url}/realms/llamastack-demo/protocol/openid-connect/token"
    payload = {
        'client_id': 'llamastack',
        'client_secret': client_secret,
        'username': username,
        'password': password,
        'grant_type': 'password'
    }
    
    response = requests.post(token_url, data=payload, verify=True)
    response.raise_for_status()
    
    token_data = response.json()
    api_key = token_data.get('access_token')
    
    print(f"✓ Authentication successful")
    print(f"  Token type: {token_data.get('token_type', 'Bearer')}")
    print(f"  Expires in: {token_data.get('expires_in', 'unknown')} seconds")

# Initialize OpenAI client
client = OpenAI(
    base_url=f"{llamastack_url}/v1",
    api_key=api_key,
)

In [ ]:
# Send a simple chat completion request
print("\nSending chat completion request...")
print("Prompt: 'Say hello from a Jupyter notebook!'")

response = client.chat.completions.create(
    model="vllm-inference/llama-3-2-3b",
    messages=[
        {"role": "user", "content": "Say hello from a Jupyter notebook!"}
    ],
    max_tokens=100
)

print("\nResponse:", response.choices[0].message.content)

In [ ]:
# Verify response is not empty
assert response.choices[0].message.content, "Expected non-empty response"
print("\n✓ Notebook demo completed successfully")
print(f"  Model: {response.model}")
print(f"  Tokens: {response.usage.total_tokens} total")